# KrVoiceAI · Colab GPU Worker

这台 Colab 当 **GPU worker**:连 `krvoice.cicy-ai.com` 队列,拉任务 → 出片 → 回传。用户在口播工坊提交,这里自动领走。素材走 OSS,不用 Google Drive。

## 用法(从上往下,一格一格跑)
1. **第1格** 确认 GPU(要 L4 / A100,24G)
2. **第2格** 拉最新代码
3. **第3格** 装环境(clone LatentSync + 依赖 + 模型,首次 15-25 分钟,**装一次复用**)
4. **第4格** 起 worker(前台跑,**日志实时滚在格子里**,一直转别停)
5. 去 **krvoice.cicy-ai.com** 选素材+文案 → 生成,回这里看日志出片

**先选 GPU**:`代码执行程序 → 更改运行时类型 → A100 或 L4`。

In [ ]:
# 1. 确认 GPU(必须 L4 或 A100,显存 ≥23G;显示 command not found = 没选 GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 拉最新代码
%cd /content
import os
if os.path.isdir('KrVoiceAI'):
    %cd KrVoiceAI
    !git pull -q
else:
    !git clone -q https://github.com/cicy-ai/KrVoiceAI.git
    %cd KrVoiceAI
!git log --oneline -1

In [ ]:
# 3. 先装好环境:LatentSync(数字人对口型)+ CosyVoice(克隆你的声)+ 依赖 + 模型
#    首次约 20-30 分钟,装在固定位置 /content 下,装一次、之后所有出片复用,不重装。
#    日志实时滚在本格;看到两个「✅ ...环境就绪」就装好了,再跑第4格。
!cd /content/KrVoiceAI && SETUP_ONLY=1 bash colab/latentsync.sh
!cd /content/KrVoiceAI && SETUP_ONLY=1 bash colab/clone.sh

In [ ]:
# 4. 起 worker(前台跑!日志实时滚到本格,worker 领任务/下载/出片每一行你都看得到)
#    这一格会「一直运行」= worker 在轮询队列,别等它结束;想停就点本格的停止方块。
#    worker 自动认 GPU:有 GPU→真出片(LatentSync 数字人);没 GPU→假出片(验链路)。命令固定,不用改。
!cd /content/KrVoiceAI && pip install -q requests edge-tts 2>&1 | tail -1
!cd /content/KrVoiceAI && QUEUE_URL=https://krvoice.cicy-ai.com python -u colab/pull_worker.py

---
- 第3格环境装一次就够,同一运行时里重跑第4格不用再装。
- 改代码后:重跑**第2格**(git pull)+ **第4格**(重启 worker)。
- 运行时被回收(闲置/超时)后要从第1格重来;环境要重装(第3格)。